# Remove Bands

[![github](https://img.shields.io/badge/-see%20sources-white?logo=github&labelColor=555)](https://github.com/gee-community/geetools/blob/main/docs/usage/removeBands.ipynb)
[![colab](https://img.shields.io/badge/-open%20in%20colab-blue?logo=googlecolab&labelColor=555)](https://colab.research.google.com/github/gee-community/geetools/blob/main/docs/usage/removeBands.ipynb)

## Getting started

This notebook demonstrates how to remove selected bands from an Earth Engine image using `geetools`. The same operation can also be applied to every image in an `ee.ImageCollection`.

### Install `geetools` when running in Colab

The following cell checks whether the notebook is running in Google Colab. If so, it installs `geetools` only when the package is not already available. This keeps the setup lightweight in other environments, such as a local Jupyter notebook.

In [ ]:
import sys
if "google.colab" in sys.modules and "geetools" not in sys.modules:
    !pip install geetools



### Import the libraries and initialize Earth Engine

`geetools` extends Earth Engine objects with the `.geetools` namespace. The initialization block first tries to use an existing Earth Engine session; if that fails, it starts the authentication flow and initializes a new session.

In [ ]:
import ee, geetools
from ee.ee_exception import EEException

from geetools.utils import initialize_documentation
try:
    ee.Initialize()
except EEException:
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize()


### Load an image and select its bands

We use one Sentinel-2 image as a compact example. The regular expression `'B.'` selects the bands whose names start with `B`; this gives us a manageable set of bands to inspect and modify.

In [ ]:
i = ee.Image('COPERNICUS/S2_HARMONIZED/20181122T142749_20181122T143353_T18GYT').select('B.')

### Inspect the original band names

Earth Engine objects are evaluated lazily. Calling `.getInfo()` requests the server-side result and brings this small list to Python, where `print` displays it. This gives us a baseline for comparing the image after removing bands.

In [ ]:
print(i.bandNames().getInfo())

## Remove selected bands

The `geetools.remove` method accepts a list of band names and returns a new image without those bands. Here we remove `B1`, `B2`, and `B7`; the original image `i` remains unchanged.

In [ ]:
removed = i.geetools.remove(['B1', 'B2', 'B7'])

### Verify the result

The resulting image should contain all the original bands except the three selected for removal. Printing the band names is a simple way to verify the transformation.

In [ ]:
print(removed.bandNames().getInfo())

## Apply the operation to an image collection

The same band-removal operation can be mapped over an `ee.ImageCollection`. Earth Engine applies the function server-side to each image, producing a new collection while preserving the original one.

### Create a small collection for demonstration

We limit the Sentinel-2 collection to five images so that its band structure can be inspected without requesting an unnecessarily large amount of metadata.

In [ ]:
col = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').limit(5)

### Helper function: print the bands of every image

An `ee.ImageCollection` contains server-side objects, so this helper calls `.getInfo()` on the small example collection, extracts each image's band metadata, and prints the band identifiers. It is intended for inspection and debugging, not for processing large collections.

In [ ]:
def print_bands(col):
    info = col.getInfo()
    images = info['features']
    for image in images:
        bands = image['bands']
        print([band['id'] for band in bands])

### Inspect the collection before editing it

This first call shows the band layout of the original images. It provides a direct before-and-after comparison for the mapped operation below.

In [ ]:
print_bands(col)

### Remove the bands from every image

`map` applies the same function to each image in the collection. The lambda calls `geetools.remove` with the same three band names, returning a new collection with the unwanted bands removed from every image.

In [ ]:
removed_col = col.map(lambda img: img.geetools.remove(['B1', 'B2', 'B7']))

### Verify the edited collection

Finally, we inspect the transformed collection. Each printed list should no longer contain `B1`, `B2`, or `B7`, confirming that the operation was applied consistently across all images.

In [ ]:
print_bands(removed_col)